# Análise: onde intervir para recuperar capacidade de cais

Notebook de análise do MVP. Ano de referência **2025**.

Ele parte da camada **gold** já construída por `scripts/pipeline.py` e reproduz,
célula a célula, as respostas às perguntas de negócio P1 a P4 e a verificação
de qualidade P5.

Ordem de execução do projeto:

1. `scripts/coleta.py` baixa os arquivos e grava o manifesto
2. `scripts/pipeline.py` constrói bronze, silver e gold
3. `scripts/qualidade.py` audita os 59 atributos
4. **este notebook** responde às perguntas
5. `notebooks/01_pipeline_databricks.py` replica tudo em PySpark na nuvem

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

PROJ = Path("..") if Path("../dados/gold").exists() else Path(".")
ANO = 2025
G = PROJ / "dados" / "gold"

fato   = pd.read_parquet(G / f"fato_atracacao_{ANO}.parquet")
dim_i  = pd.read_parquet(G / f"dim_instalacao_{ANO}.parquet")
dim_b  = pd.read_parquet(G / f"dim_berco_{ANO}.parquet")
dim_t  = pd.read_parquet(G / f"dim_tempo_{ANO}.parquet")
dim_n  = pd.read_parquet(G / f"dim_navegacao_{ANO}.parquet")
dim_c  = pd.read_parquet(G / f"dim_natureza_carga_{ANO}.parquet")

print(f"fato_atracacao ...... {len(fato):>8,} linhas, {len(fato.columns)} atributos")
for nome, d in [("dim_instalacao", dim_i), ("dim_berco", dim_b), ("dim_tempo", dim_t),
                ("dim_navegacao", dim_n), ("dim_natureza_carga", dim_c)]:
    print(f"{nome:.<20} {len(d):>8,} linhas, {len(d.columns)} atributos")

fato_atracacao ......  116,327 linhas, 28 atributos
dim_instalacao......      212 linhas, 13 atributos
dim_berco...........      625 linhas, 3 atributos
dim_tempo...........      365 linhas, 8 atributos
dim_navegacao.......       35 linhas, 4 atributos
dim_natureza_carga..        5 linhas, 3 atributos


## Junção do esquema estrela

A tabela fato guarda apenas chaves e medidas. Os atributos descritivos vêm das
dimensões. Esta é a junção usada em toda a análise.

In [2]:
base = (fato
        .merge(dim_i, on="sk_instalacao", how="left")
        .merge(dim_n, on="sk_navegacao",  how="left")
        .merge(dim_c, on="sk_natureza",   how="left"))

# Só entram na análise de tempo os registros aptos: sem cronologia inválida,
# sem estadia outlier e com tempo atracado definido.
base = base[base["flag_apto_analise_tempo"]]
print(f"{len(base):,} atracações aptas de {len(fato):,} ({len(base)/len(fato)*100:.2f}%)")

# Segmentação por modo de navegação. A justificativa está na P2.
SEGMENTOS = {"Marítimo de carga": ["Longo Curso", "Cabotagem"],
             "Interior (fluvial)": ["Interior"],
             "Apoio": ["Apoio Marítimo", "Apoio Portuário"]}
base["segmento"] = "Outros"
for nome, tipos in SEGMENTOS.items():
    base.loc[base["tipo_navegacao"].isin(tipos), "segmento"] = nome

base["segmento"].value_counts()

115,071 atracações aptas de 116,327 (98.92%)


segmento
Interior (fluvial)    53785
Marítimo de carga     43389
Apoio                 17897
Name: count, dtype: int64

## P1. Para onde vai o tempo da embarcação

A decomposição oficial da ANTAQ separa quatro tempos. T2 e T4 são berço
ocupado sem nada sendo movimentado: é a capacidade mais barata de recuperar.

O ponto de atenção é o estimador. Soma e mediana levam a conclusões opostas.

In [3]:
ETAPAS = ["t1_espera_atracacao_h", "t2_espera_inicio_op_h",
          "t3_operacao_h", "t4_espera_desatracacao_h"]
comp = base.dropna(subset=ETAPAS)

somas = comp[ETAPAS].sum()
meds  = comp[ETAPAS].median()

p1 = pd.DataFrame({
    "horas_no_ano": somas.round(0),
    "share_por_soma_%": (somas / somas.sum() * 100).round(2),
    "atracação_mediana_h": meds.round(2),
    "share_por_mediana_%": (meds / meds.sum() * 100).round(2),
})
p1.index = ["T1 espera para atracar", "T2 espera p/ iniciar",
            "T3 operação", "T4 espera p/ sair"]
print(f"navio-hora total: {somas.sum():,.0f} h em {len(comp):,} atracações\n")
p1

navio-hora total: 6,575,336 h em 108,506 atracações



,horas_no_ano,share_por_soma_%,atracação_mediana_h,share_por_mediana_%
T1 espera para atracar,3585367.0,54.53,1.00,10.77
T2 espera p/ iniciar,412872.0,6.28,0.37,3.95
T3 operação,2196792.0,33.41,7.00,75.40
T4 espera p/ sair,380306.0,5.78,0.92,9.87


In [4]:
t1 = comp["t1_espera_atracacao_h"]
print(f"assimetria de T1 .... {t1.skew():.2f}")
print(f"média ............... {t1.mean():.2f} h")
print(f"mediana ............. {t1.median():.2f} h")
print(f"percentil 99 ........ {t1.quantile(0.99):.1f} h\n")

for q in [0.90, 0.95, 0.99]:
    acima = t1[t1 > t1.quantile(q)]
    print(f"os {(1-q)*100:>4.0f}% com maior espera concentram "
          f"{acima.sum()/t1.sum()*100:5.1f}% de todas as horas de espera")

improd = somas["t2_espera_inicio_op_h"] + somas["t4_espera_desatracacao_h"]
atracado = improd + somas["t3_operacao_h"]
print(f"\nberço improdutivo: {improd:,.0f} h = {improd/atracado*100:.2f}% do tempo atracado")
print(f"equivalente a {improd/8760:.1f} posições de cais paradas o ano inteiro")

assimetria de T1 .... 3.88
média ............... 33.04 h
mediana ............. 1.00 h
percentil 99 ........ 411.0 h

os   10% com maior espera concentram  68.4% de todas as horas de espera
os    5% com maior espera concentram  47.8% de todas as horas de espera
os    1% com maior espera concentram  15.4% de todas as horas de espera

berço improdutivo: 793,178 h = 26.53% do tempo atracado
equivalente a 90.5 posições de cais paradas o ano inteiro


## P2. Onde está concentrado, e por que o ranking direto engana

Primeiro o ranking sem segmentação, que é o que sairia de uma leitura direta
do indicador. Depois o teste da hipótese alternativa de que ele esteja
capturando modelo de negócio em vez de ineficiência.

In [5]:
ingenuo = (comp.groupby(["sk_instalacao", "porto_atracacao", "sguf"])
           .agg(atracacoes=("sk_atracacao", "count"),
                improdutivo_h=("berco_improdutivo_h", "sum"),
                atracado_h=("ta_atracado_h", "sum"))
           .reset_index())
ingenuo["pct_improdutivo"] = (ingenuo["improdutivo_h"] / ingenuo["atracado_h"] * 100).round(1)
ingenuo.nlargest(8, "improdutivo_h")[
    ["porto_atracacao", "sguf", "atracacoes", "improdutivo_h", "pct_improdutivo"]].round(0)

,porto_atracacao,sguf,atracacoes,improdutivo_h,pct_improdutivo
100,Cargill Agrícola S.A. (ETC Miritituba/PA),PA,1084,145398.0,97.0
201,Santos,SP,5043,41313.0,16.0
198,LDC Pederneiras,SP,852,27635.0,85.0
159,F. H. de Oliveira Peixoto,RO,93,27520.0,85.0
64,Santana,AP,560,23030.0,62.0
71,Niterói,RJ,547,21909.0,98.0
135,Brasil Logística Offshore e Estaleiro Naval,RJ,9855,18201.0,14.0
204,Vila do Conde,PA,2113,17758.0,23.0


In [6]:
# Perfil operacional das instalações do topo: o teste que derruba a leitura direta.
for cod in ingenuo.nlargest(4, "improdutivo_h")["sk_instalacao"]:
    s = comp[comp["sk_instalacao"] == cod]
    print(f"--- {s['porto_atracacao'].iloc[0]} ({s['sguf'].iloc[0]}), n={len(s):,}")
    print(f"    navegação .......... {s['tipo_navegacao'].value_counts().head(2).to_dict()}")
    print(f"    instalação em rio .. {s['instalacao_em_rio'].value_counts().to_dict()}")
    print(f"    medianas ........... T1={s['t1_espera_atracacao_h'].median():.1f}h  "
          f"T2={s['t2_espera_inicio_op_h'].median():.1f}h  "
          f"T3={s['t3_operacao_h'].median():.1f}h  "
          f"T4={s['t4_espera_desatracacao_h'].median():.1f}h")
    print(f"    peso médio ......... {s['peso_bruto_t'].mean():,.0f} t\n")

--- Cargill Agrícola S.A. (ETC Miritituba/PA) (PA), n=1,084
    navegação .......... {'Interior': 1084}
    instalação em rio .. {'Sim': 1084}
    medianas ........... T1=0.0h  T2=44.0h  T3=2.5h  T4=43.5h
    peso médio ......... 2,812 t

--- Santos (SP), n=5,043
    navegação .......... {'Longo Curso': 4112, 'Cabotagem': 864}
    instalação em rio .. {'Não': 5043}
    medianas ........... T1=19.7h  T2=2.0h  T3=29.0h  T4=4.1h
    peso médio ......... 27,957 t

--- LDC Pederneiras (SP), n=852
    navegação .......... {'Interior': 852}
    instalação em rio .. {'Sim': 852}
    medianas ........... T1=0.0h  T2=14.1h  T3=4.7h  T4=9.8h
    peso médio ......... 1,343 t

--- F. H. de Oliveira Peixoto (RO), n=93
    navegação .......... {'Interior': 92, 'Apoio Portuário': 1}
    instalação em rio .. {'Sim': 93}
    medianas ........... T1=1.0h  T2=241.4h  T3=33.5h  T4=46.0h
    peso médio ......... 1,207 t



O padrão é claro. As instalações do topo são terminais fluviais de barcaça,
com espera para atracar perto de zero, operação curta e longa espera atracado.
A barcaça atraca e aguarda a formação do comboio: a espera atracada **é** o
modelo logístico, não desperdício.

Comparando os segmentos por volume movimentado, a diferença fica evidente.

In [7]:
seg = (comp.groupby("segmento")
       .agg(atracacoes=("sk_atracacao", "count"),
            peso_Mt=("peso_bruto_t", lambda x: x.sum() / 1e6),
            improdutivo_h=("berco_improdutivo_h", "sum"),
            operacao_h=("t3_operacao_h", "sum"),
            atracado_h=("ta_atracado_h", "sum"))
       .reset_index())
seg["pct_improdutivo"] = (seg["improdutivo_h"] / seg["atracado_h"] * 100).round(2)
seg["t_por_atracacao"] = (seg["peso_Mt"] * 1e6 / seg["atracacoes"]).round(0)
seg.round(2)

,segmento,atracacoes,peso_Mt,improdutivo_h,operacao_h,atracado_h,pct_improdutivo,t_por_atracacao
0,Apoio,13084,9.73,54301.40,209024.40,263325.80,20.62,743.0
1,Interior (fluvial),52642,87.07,422185.97,466731.08,888917.05,47.49,1654.0
2,Marítimo de carga,42780,1282.44,316690.33,1521036.27,1837726.60,17.23,29977.0


In [8]:
# Ranking corrigido: só o universo em que berço improdutivo significa
# capacidade de cais marítimo perdida.
mar = comp[(comp["segmento"] == "Marítimo de carga") & comp["operacao_produtiva"]]

rank = (mar.groupby(["sk_instalacao", "porto_atracacao", "sguf", "tipo_autoridade"])
        .agg(atracacoes=("sk_atracacao", "count"),
             improdutivo_h=("berco_improdutivo_h", "sum"),
             atracado_h=("ta_atracado_h", "sum"),
             peso_t=("peso_bruto_t", "sum"))
        .reset_index())
rank["pct_improdutivo"] = (rank["improdutivo_h"] / rank["atracado_h"] * 100).round(2)
rank = rank[rank["atracacoes"] >= 50]

alvo = rank["pct_improdutivo"].median()
rank["horas_recuperaveis"] = np.where(
    rank["pct_improdutivo"] > alvo,
    rank["improdutivo_h"] - rank["atracado_h"] * alvo / 100, 0.0)
rank = rank.sort_values("horas_recuperaveis", ascending=False)

print(f"universo: {len(rank)} instalações, {rank['atracacoes'].sum():,} atracações")
print(f"alvo (mediana do setor): {alvo:.2f}% do tempo atracado")
print(f"recuperável: {rank['horas_recuperaveis'].sum():,.0f} h = "
      f"{rank['horas_recuperaveis'].sum()/8760:.2f} posições de cais")
print(f"concentração no top 10: "
      f"{rank.head(10)['horas_recuperaveis'].sum()/rank['horas_recuperaveis'].sum()*100:.1f}%\n")
rank.head(10)[["porto_atracacao", "sguf", "atracacoes", "pct_improdutivo",
               "horas_recuperaveis"]].round(1)

universo: 76 instalações, 30,772 atracações
alvo (mediana do setor): 18.23% do tempo atracado
recuperável: 46,095 h = 5.26 posições de cais
concentração no top 10: 71.3%



,porto_atracacao,sguf,atracacoes,pct_improdutivo,horas_recuperaveis
8,Terminal Graneleiro Hermasa,AM,147,44.6,10038.8
82,Terminal de Petróleo TPET/TOIL - Açu,RJ,384,45.1,5648.6
67,Terminal Aquaviário de Angra dos Reis,RJ,486,35.9,4013.4
117,Suape,PE,1572,24.4,3501.0
111,Terminal Aquaviário de São Sebastião (Almirant...,SP,711,28.0,2577.9
31,Portocel - Terminal Especializado de Barra do ...,ES,407,24.9,1671.4
68,Terminal Aquaviário da Ilha D'Água,RJ,367,26.6,1577.3
100,Porto Itapoá Terminais Portuários,SC,692,27.6,1451.4
74,Porto do Açu - Terminal de Minério,RJ,141,36.6,1195.3
17,Terminal Aquaviário de Madre de Deus,BA,380,25.2,1190.8


In [9]:
# O custo do viés, quantificado: quantas instalações sobrevivem à correção?
top_ingenuo   = set(ingenuo.nlargest(10, "improdutivo_h")["sk_instalacao"])
top_corrigido = set(rank.head(10)["sk_instalacao"])
comuns = top_ingenuo & top_corrigido
print(f"top 10 ingênuo   : {len(top_ingenuo)} instalações")
print(f"top 10 corrigido : {len(top_corrigido)} instalações")
print(f"em comum ........ : {len(comuns)}  ->  {10-len(comuns)} de 10 decisões "
      f"seriam direcionadas ao lugar errado")

top 10 ingênuo   : 10 instalações
top 10 corrigido : 10 instalações
em comum ........ : 1  ->  9 de 10 decisões seriam direcionadas ao lugar errado


## P3. Fila ou coordenação?

A decisão entre construir berço e reorganizar programação depende de a espera
ser ou não explicada por congestionamento.

Antes de calcular ocupação, é preciso resolver um problema: `IDBerco` não é
uma posição exclusiva de atracação.

In [10]:
silver = pd.read_parquet(PROJ / "dados" / "silver" / f"atracacao_tratada_{ANO}.parquet")
s = silver.dropna(subset=["Data Atracação", "Data Desatracação"])
s = s[~s["flag_cronologia_invalida"] & ~s["flag_estadia_outlier"]]

# Ocupação simultânea máxima observada em cada berço ao longo do ano.
ev = pd.concat([
    pd.DataFrame({"b": s["IDBerco"], "t": s["Data Atracação"],    "d":  1}),
    pd.DataFrame({"b": s["IDBerco"], "t": s["Data Desatracação"], "d": -1}),
]).sort_values(["b", "t", "d"], ascending=[True, True, False])
ev["oc"] = ev.groupby("b")["d"].cumsum()
pos = ev.groupby("b")["oc"].max().clip(lower=1)

print(f"berços avaliados ................. {len(pos):,}")
print(f"com posição única (exclusivos) ... {(pos<=1).sum():,} ({(pos<=1).mean()*100:.1f}%)")
print(f"mediana de posições .............. {pos.median():.0f}")
print(f"percentil 90 ..................... {pos.quantile(0.9):.0f}")
print(f"máximo ........................... {pos.max()}")
print("\nSupor uma posição por berço produziria ocupação acima de 100% "
      "e falso diagnóstico de congestionamento.")

berços avaliados ................. 624
com posição única (exclusivos) ... 219 (35.1%)
mediana de posições .............. 2
percentil 90 ..................... 5
máximo ........................... 125

Supor uma posição por berço produziria ocupação acima de 100% e falso diagnóstico de congestionamento.


In [11]:
pos_inst = (dim_b.merge(pos.rename("posicoes"), left_on="sk_berco", right_index=True, how="left")
            .groupby("sk_instalacao")["posicoes"].sum().rename("posicoes_inst").reset_index())

u = rank.merge(pos_inst, on="sk_instalacao", how="left")
u["posicoes_inst"] = u["posicoes_inst"].fillna(1)
u["ocupacao_pct"] = (u["atracado_h"] / (8760 * u["posicoes_inst"]) * 100).round(2)
u["espera_mediana_h"] = (mar.groupby("sk_instalacao")["t1_espera_atracacao_h"]
                         .median().reindex(u["sk_instalacao"]).values)

uu = u.dropna(subset=["ocupacao_pct", "espera_mediana_h"])
rho, p = stats.spearmanr(uu["ocupacao_pct"], uu["espera_mediana_h"])
print(f"n = {len(uu)} instalações marítimas")
print(f"Spearman rho = {rho:.4f}  (p = {p:.2e})")
print(f"variação de ordem explicada: {rho**2*100:.1f}%")
print(f"ocupação mediana do setor: {uu['ocupacao_pct'].median():.2f}%\n")
print("Leitura: a relação existe e é significativa, mas explica pouco. "
      "Ocupação na casa dos 20% não é sistema saturado. Para a maioria "
      "das instalações da lista, berço novo não é a intervenção indicada.")

n = 76 instalações marítimas
Spearman rho = 0.3956  (p = 4.05e-04)
variação de ordem explicada: 15.6%
ocupação mediana do setor: 21.44%

Leitura: a relação existe e é significativa, mas explica pouco. Ocupação na casa dos 20% não é sistema saturado. Para a maioria das instalações da lista, berço novo não é a intervenção indicada.


## P4. Intervir por terminal ou por tipo de carga?

Decomposição da variância do logaritmo da prancha operacional. A pergunta é
qual fator explica mais: quem é o terminal, ou o que ele movimenta.

In [12]:
prod = base[base["operacao_produtiva"]
            & base["prancha_operacional_th"].notna()
            & ~base["flag_prancha_implausivel"]
            & base["flag_atracacao_exclusiva"].fillna(False)].copy()
prod["log_prancha"] = np.log(prod["prancha_operacional_th"])

def eta2(df, grupo, alvo="log_prancha"):
    m = df[alvo].mean()
    sqt = ((df[alvo] - m) ** 2).sum()
    g = df.groupby(grupo)[alvo].agg(["mean", "count"])
    return float((g["count"] * (g["mean"] - m) ** 2).sum() / sqt)

print(f"atracações na análise: {len(prod):,}\n")
for rot, col in [("qual instalação", "sk_instalacao"),
                 ("qual tipo de carga", "natureza_carga"),
                 ("qual navegação", "tipo_navegacao")]:
    print(f"{rot:.<24} {eta2(prod, col)*100:5.1f}% da variação explicada")

atracações na análise: 78,618

qual instalação.........  59.8% da variação explicada
qual tipo de carga......  18.9% da variação explicada
qual navegação..........   5.4% da variação explicada


In [13]:
# Dispersão entre terminais DENTRO de cada tipo de carga.
linhas = []
for nat, g in prod.groupby("natureza_carga"):
    med = g.groupby("sk_instalacao")["prancha_operacional_th"].median()
    med = med[g.groupby("sk_instalacao").size() >= 30]
    if len(med) >= 5:
        linhas.append({"natureza_carga": nat, "instalacoes": len(med),
                       "p10_t_h": round(med.quantile(0.10), 1),
                       "p90_t_h": round(med.quantile(0.90), 1),
                       "razao_p90_p10": round(med.quantile(0.90) / med.quantile(0.10), 1)})
pd.DataFrame(linhas).sort_values("razao_p90_p10", ascending=False)

,natureza_carga,instalacoes,p10_t_h,p90_t_h,razao_p90_p10
2,Granel Líquido e Gasoso,60,48.9,1022.3,20.9
3,Granel Sólido,78,137.3,2347.4,17.1
1,Carga Geral,40,33.0,357.4,10.8
0,Carga Conteinerizada,22,244.3,1267.0,5.2


## P5. A base sustenta essas decisões?

Resumo da auditoria. O relatório completo, com os 59 atributos, está em
`outputs/relatorio_qualidade_{ANO}.md`.

In [14]:
qa = pd.read_csv(PROJ / "outputs" / f"qualidade_por_atributo_{ANO}.csv")
print(f"{len(qa)} atributos auditados em {qa['tabela'].nunique()} tabelas\n")
print(qa.groupby("situacao").size().to_string())
print()
print("Atributos com menor completude:")
print(qa.nsmallest(5, "completude_pct")[["tabela", "atributo", "completude_pct"]].to_string(index=False))
print()
print("Violações de domínio declarado:")
v = qa[qa["violacoes_de_dominio"] > 0][["tabela", "atributo", "violacoes_de_dominio", "regra_de_dominio"]]
print(v.to_string(index=False) if len(v) else "  nenhuma")

59 atributos auditados em 6 tabelas

situacao
conforme                  36
conforme com ausências    15
violação de domínio        8

Atributos com menor completude:
        tabela               atributo  completude_pct
dim_instalacao     apelido_instalacao           32.08
dim_instalacao    regiao_hidrografica           47.64
fato_atracacao             numero_imo           51.49
fato_atracacao prancha_operacional_th           80.96
fato_atracacao       prancha_geral_th           80.96

Violações de domínio declarado:
        tabela               atributo  violacoes_de_dominio          regra_de_dominio
fato_atracacao  t1_espera_atracacao_h                   144  esperado entre 0 e 8,760
fato_atracacao  t2_espera_inicio_op_h                     3  esperado entre 0 e 8,760
fato_atracacao          t3_operacao_h                     1  esperado entre 0 e 8,760
fato_atracacao          ta_atracado_h                     8  esperado entre 0 e 8,760
fato_atracacao           te_estadia_h          

In [15]:
rel = json.load(open(PROJ / "outputs" / f"relatorio_qualidade_{ANO}.json", encoding="utf-8"))
print("Integridade referencial e cronológica")
print(json.dumps(rel["consistencia"], ensure_ascii=False, indent=2))

Integridade referencial e cronológica
{
  "fk_instalacao_orfa": 0,
  "fk_berco_orfa": 6,
  "fk_data_orfa": 0,
  "fk_navegacao_orfa": 6,
  "fk_natureza_orfa": 0,
  "fk_berco_para_instalacao_orfa": 0,
  "registros_com_cronologia_invalida": 0,
  "quebras_por_etapa": {
    "atracacao_antes_da_chegada": 0,
    "inicio_op_antes_da_atracacao": 0,
    "termino_op_antes_do_inicio": 0,
    "desatracacao_antes_do_termino": 0,
    "desatracacao_antes_da_atracacao": 0
  },
  "coerencia_T2_T3_T4_igual_TA": {
    "registros_testados": 109632,
    "divergencias_acima_de_1_segundo": 0,
    "erro_maximo_horas": 0.0
  }
}


## Síntese para a decisão

A recomendação não é a mesma para todos os terminais da lista, e é a
decomposição do tempo que a diferencia.

Onde **T2 e T4** dominam, a intervenção é de coordenação: liberação
documental, escala de turno, sequenciamento de equipe. Custa pouco e é rápida.

Onde **T3** domina e a prancha está no percentil baixo do seu tipo de carga,
a intervenção é de equipamento, e aí há capital envolvido.

Onde **T1** domina mas a ocupação é baixa, o caminho é programação de janelas,
não obra civil.

Esse é o produto do MVP: ele não diz quanto investir, mas elimina classes
inteiras de intervenção para cada terminal, que é o que um sistema de suporte
à decisão deve fazer.

In [16]:
top = rank.head(10).merge(u[["sk_instalacao", "ocupacao_pct"]], on="sk_instalacao", how="left")
perfil = mar.groupby("sk_instalacao")[["t1_espera_atracacao_h", "t2_espera_inicio_op_h",
                                       "t3_operacao_h", "t4_espera_desatracacao_h"]].median()
top = top.merge(perfil, on="sk_instalacao", how="left")

def indicar(r):
    if r["ocupacao_pct"] is not None and r["t1_espera_atracacao_h"] > r["t3_operacao_h"] and r["ocupacao_pct"] < 40:
        return "programação de janelas"
    if (r["t2_espera_inicio_op_h"] + r["t4_espera_desatracacao_h"]) > r["t3_operacao_h"]:
        return "coordenação em terra"
    return "produtividade de equipamento"

top["intervenção_indicada"] = top.apply(indicar, axis=1)
top[["porto_atracacao", "sguf", "atracacoes", "pct_improdutivo",
     "horas_recuperaveis", "intervenção_indicada"]].round(1)

,porto_atracacao,sguf,atracacoes,pct_improdutivo,horas_recuperaveis,intervenção_indicada
0,Terminal Graneleiro Hermasa,AM,147,44.6,10038.8,produtividade de equipamento
1,Terminal de Petróleo TPET/TOIL - Açu,RJ,384,45.1,5648.6,produtividade de equipamento
2,Terminal Aquaviário de Angra dos Reis,RJ,486,35.9,4013.4,programação de janelas
3,Suape,PE,1572,24.4,3501.0,produtividade de equipamento
4,Terminal Aquaviário de São Sebastião (Almirant...,SP,711,28.0,2577.9,produtividade de equipamento
5,Portocel - Terminal Especializado de Barra do ...,ES,407,24.9,1671.4,produtividade de equipamento
6,Terminal Aquaviário da Ilha D'Água,RJ,367,26.6,1577.3,programação de janelas
7,Porto Itapoá Terminais Portuários,SC,692,27.6,1451.4,produtividade de equipamento
8,Porto do Açu - Terminal de Minério,RJ,141,36.6,1195.3,produtividade de equipamento
9,Terminal Aquaviário de Madre de Deus,BA,380,25.2,1190.8,produtividade de equipamento
